# Linear Regression using Mini-batch Gradient Descent

Mini-batch GD is between GD and SGD: we split the training data into small batches (here 32 samples) and update after each batch. It is less noisy than SGD and does many more updates per epoch than GD.

We compare it with and without momentum (beta = 0.9).

In [ ]:
import os
import json
import time
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch                     # only used for the optimizers (torch.optim)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

## Dataset
We used the **Consumer Airfare Report (Table 6)** from the U.S. Department of Transportation. It has the prices of domestic flights in the USA for every route between two cities (with at least 10 passengers per day), every quarter of the year, from the 1990s until now. It has hundreds of thousands of rows.

The target is `fare`, the average one-way price of a flight on that route in that quarter, so this is a **regression** problem. The price is not a fixed formula: it depends on the distance, but also on how many people fly on the route, how much competition there is between the airlines, which airlines fly there, and the time (year and quarter).

Preprocessing steps:
- we made all column names lowercase and removed rows with missing or impossible values
- features: distance (`nsmiles`), log of passengers per day (passengers is very skewed), market share of the biggest airline (`large_ms`), market share of the cheapest airline (`lf_ms`), year, quarter, and which airline is the biggest and which is the cheapest on the route (one-hot, the 8 most common airlines and the rest as "Other")
- we did **not** use `fare_lg` and `fare_low` because they are prices themselves (data leakage)
- time based split: the last 2 years are the test set, all the years before are the training set
- StandardScaler only on the 5 continuous columns (fit on the training data only)

In [ ]:
url = "https://data.transportation.gov/api/views/yj5y-b2ir/rows.csv?accessType=DOWNLOAD"
file_name = "airfare_table6.csv"

if not os.path.exists(file_name):
    print("downloading the file, it can take a minute...")
    urllib.request.urlretrieve(url, file_name)

df = pd.read_csv(file_name, low_memory=False)
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")     # same column names style everywhere
original_rows = len(df)

print("number of rows:", original_rows)
print("years:", df["year"].min(), "to", df["year"].max())
df.head()

In [ ]:
cols = ["year", "quarter", "nsmiles", "passengers", "fare", "large_ms", "lf_ms", "carrier_lg", "carrier_low"]
df = df[cols].copy()

# make sure the numbers are numbers
for c in ["year", "quarter", "nsmiles", "passengers", "fare", "large_ms", "lf_ms"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# cleaning: remove missing and impossible values
df = df.dropna()
df = df[df["fare"].between(20, 2000)]
df = df[df["nsmiles"] > 0]
df = df[df["passengers"] > 0]
df = df[df["large_ms"].between(0, 1) & df["lf_ms"].between(0, 1)]

# keep the 8 most common airlines, all the others become "Other"
for c in ["carrier_lg", "carrier_low"]:
    top = df[c].value_counts().index[:8]
    df[c] = df[c].where(df[c].isin(top), "Other")

print("rows after cleaning:", len(df))
print("rows removed: {:.1f}%".format(100 * (1 - len(df) / original_rows)))
df[["fare", "nsmiles", "passengers", "large_ms", "lf_ms"]].describe()

In [ ]:
sample = df.sample(5000, random_state=42)
per_year = df.groupby("year")["fare"].mean()

plt.figure(figsize=(17, 4.5))

plt.subplot(1, 3, 1)
plt.hist(df["fare"], bins=50, color="steelblue")
plt.xlabel("average fare ($)")
plt.ylabel("number of rows")
plt.title("Distribution of the fare")

plt.subplot(1, 3, 2)
plt.scatter(sample["nsmiles"], sample["fare"], s=5, alpha=0.4)
plt.xlabel("distance (miles)")
plt.ylabel("average fare ($)")
plt.title("Fare vs distance (5000 random rows)")

plt.subplot(1, 3, 3)
plt.plot(per_year.index, per_year.values, marker="o", color="darkorange")
plt.xlabel("year")
plt.ylabel("average fare ($)")
plt.title("The price changes over the years")
plt.grid(True)

plt.tight_layout()
plt.savefig("minibatch_data.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# one-hot encoding for the categorical columns
df = pd.get_dummies(df, columns=["quarter", "carrier_lg", "carrier_low"], drop_first=True, dtype=int)

df["log_passengers"] = np.log(df["passengers"])     # passengers is very skewed (10 to thousands), so we use the log

feature_cols = ["nsmiles", "log_passengers", "large_ms", "lf_ms", "year"]
feature_cols += [c for c in df.columns if c.startswith(("quarter_", "carrier_lg_", "carrier_low_"))]

X = df[feature_cols].values.astype(float)
y = df["fare"].values

# time based split: the last 2 years are the test set
last_year = df["year"].max()
train_mask = (df["year"] <= last_year - 2).values
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

# scale only the 5 continuous columns, the 0/1 columns stay as they are
scaler = StandardScaler()
X_train[:, :5] = scaler.fit_transform(X_train[:, :5])
X_test[:, :5] = scaler.transform(X_test[:, :5])

print("number of features:", len(feature_cols))
print("train:", X_train.shape, " test:", X_test.shape)
print("test years:", int(last_year - 1), "and", int(last_year))

## Best possible solution (Normal Equation)
Linear regression has an exact solution, so we can compute the best weights directly. We use `lstsq` instead of the normal inverse because some columns are almost dependent.

We will use this result as a **reference**: the optimizers are trying to reach this solution, and a good optimizer should get very close to it.

In [ ]:
Xb = np.hstack([X_train, np.ones((len(X_train), 1))])      # add a column of ones for the bias
Xb_test = np.hstack([X_test, np.ones((len(X_test), 1))])

t0 = time.time()
w_exact = np.linalg.lstsq(Xb, y_train, rcond=None)[0]
exact_time = time.time() - t0

best_loss = np.mean((Xb @ w_exact - y_train) ** 2)
pred_exact = Xb_test @ w_exact

print("Normal Equation (best possible):")
print("  train MSE :", round(best_loss, 4))
print("  test RMSE :", round(np.sqrt(mean_squared_error(y_test, pred_exact)), 4), "$")
print("  test R2   :", round(r2_score(y_test, pred_exact), 4))
print("average fare in the test set:", round(y_test.mean(), 2), "$")

## Linear Regression model (NumPy)
We wrote the model ourselves using NumPy:

- prediction: y_hat = X w + b
- loss (MSE): L = (1/n) * sum((y_hat - y)^2)
- gradients:  dL/dw = (2/n) * X^T (y_hat - y),   dL/db = (2/n) * sum(y_hat - y)

The optimizer (the update step) is taken from `torch.optim`. We compute the gradients with our NumPy model and give them to the optimizer, and the optimizer updates the weights.

In [ ]:
class LinearRegressionNP:
    def __init__(self, n_features):
        self.w = np.zeros(n_features)
        self.b = 0.0

    def predict(self, X):
        return X @ self.w + self.b

    def loss(self, X, y):
        return np.mean((self.predict(X) - y) ** 2)

    def gradients(self, X, y):
        n = len(y)
        error = self.predict(X) - y
        dw = (2 / n) * X.T @ error
        db = (2 / n) * np.sum(error)
        return dw, db

## Training

In [ ]:
EPOCHS = 20
LR = 0.01
BATCH_SIZE = 32
PATH_EVERY = 1000   # save the weights every 1000 updates (to draw the path)

In [ ]:
def train_model(lr, momentum=0.0, epochs=EPOCHS):
    np.random.seed(42)
    n, d = X_train.shape
    model = LinearRegressionNP(d)

    # the weights are torch tensors so the torch optimizer can update them (we start from zeros)
    w_param = torch.zeros(d, dtype=torch.float64)
    b_param = torch.zeros(1, dtype=torch.float64)
    optimizer = torch.optim.SGD([w_param, b_param], lr=lr, momentum=momentum)

    losses = [model.loss(X_train, y_train)]                       # loss at the start (all weights = 0)
    epochs_axis = [0.0]                                           # how many epochs passed at each logged loss
    times = [0.0]                                                 # training time at each logged loss
    path = [np.append(w_param.numpy().copy(), b_param.item())]    # starting point, used later to draw the path
    step = 0
    log_time = 0.0
    start_time = time.time()

    for epoch in range(epochs):
        perm = np.random.permutation(n)          # shuffle the data every epoch
        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            model.w = w_param.numpy()      # current weights
            model.b = b_param.item()
            dw, db = model.gradients(X_train[idx], y_train[idx])   # gradients from our NumPy model
            w_param.grad = torch.tensor(dw, dtype=torch.float64)
            b_param.grad = torch.tensor([db], dtype=torch.float64)
            optimizer.step()                  # the update is done by the library optimizer

            step += 1
            if step % PATH_EVERY == 0:        # save the weights to draw the path later
                path.append(np.append(w_param.numpy().copy(), b_param.item()))

        # loss on all the training data at the end of the epoch
        t = time.time()
        model.w = w_param.numpy()
        model.b = b_param.item()
        losses.append(model.loss(X_train, y_train))
        log_time += time.time() - t                       # we don't count the time used for logging
        times.append(time.time() - start_time - log_time)
        epochs_axis.append(epoch + 1)
        if (epoch + 1) % 5 == 0:
            print(f"epoch {epoch + 1}  loss = {losses[-1]:.4f}")

    model.time = time.time() - start_time - log_time
    model.times = times
    model.epochs = epochs_axis
    model.path = path
    return model, losses

### Mini-batch GD

In [ ]:
model_1, losses_1 = train_model(lr=LR)

### Mini-batch GD + Momentum

In [ ]:
model_2, losses_2 = train_model(lr=LR, momentum=0.9)

In [ ]:
L1 = "Mini-batch GD"
L2 = "Mini-batch GD + Momentum"
XLABEL = "epoch"

## Results

In [ ]:
rows = []
for name, m in [(L1, model_1), (L2, model_2)]:
    train_mse = np.mean((m.predict(X_train) - y_train) ** 2)
    pred = m.predict(X_test)
    rows.append({
        "method": name,
        "train MSE": train_mse,
        "gap to best (%)": 100 * (train_mse - best_loss) / best_loss,
        "test RMSE ($)": np.sqrt(mean_squared_error(y_test, pred)),
        "test R2": r2_score(y_test, pred),
        "distance to exact weights": np.linalg.norm(np.append(m.w, m.b) - w_exact),
        "time (s)": m.time,
    })
rows.append({
    "method": "Normal Equation",
    "train MSE": best_loss,
    "gap to best (%)": 0.0,
    "test RMSE ($)": np.sqrt(mean_squared_error(y_test, pred_exact)),
    "test R2": r2_score(y_test, pred_exact),
    "distance to exact weights": 0.0,
    "time (s)": exact_time,
})

results = pd.DataFrame(rows).round(4)
print(results.to_string(index=False))

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(losses_1, label=L1, linewidth=2)
plt.plot(losses_2, label=L2, linewidth=2)
plt.axhline(best_loss, color="red", linestyle="--", label="Normal Equation (best possible)")
plt.yscale("log")
plt.xlabel(XLABEL)
plt.ylabel("training MSE")
plt.title("Training loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
gap_1 = np.array(losses_1) - best_loss
gap_2 = np.array(losses_2) - best_loss
plt.plot(np.maximum(gap_1, 1e-8), label=L1, linewidth=2)
plt.plot(np.maximum(gap_2, 1e-8), label=L2, linewidth=2)
plt.yscale("log")
plt.xlabel(XLABEL)
plt.ylabel("training MSE - best MSE")
plt.title("How far are we from the best solution?")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("minibatch_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
idx = np.random.choice(len(y_test), 5000, replace=False)     # 5000 random test trips

plt.figure(figsize=(12, 5))
for k, (m, name) in enumerate([(model_1, L1), (model_2, L2)]):
    pred = m.predict(X_test)
    plt.subplot(1, 2, k + 1)
    plt.scatter(y_test[idx], pred[idx], s=5, alpha=0.4)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")   # perfect prediction
    plt.xlabel("actual fare ($)")
    plt.ylabel("predicted fare ($)")
    plt.title(f"{name}  (R2 = {r2_score(y_test, pred):.3f})")
plt.tight_layout()
plt.savefig("minibatch_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## Path of the optimizer on the loss surface
We want to see how each optimizer moves to the best solution. We have many weights, so we can't draw everything. We choose only **2 weights** (distance and log of passengers per day) and keep the other weights fixed at their best values.

- left plot: the loss surface in 3D (it is shaped like a bowl, because linear regression has only one minimum). The lines are the paths of the optimizers.
- right plot: the same thing from above. The red star is the best solution from the Normal Equation.
- we draw the **log10** of the loss, because the loss at the start is very big and without the log the bottom of the bowl would look flat.
- the paths are drawn on the surface of these 2 weights only, so the real loss at each point can be a bit higher because the other weights are also changing.

In [ ]:
def plot_path(models, labels, i=0, j=1):
    Xs = X_train[::50]            # small sample of the data to make the plot fast
    ys = y_train[::50]
    w_best = w_exact[:-1]
    b_best = w_exact[-1]

    def loss_2w(a, b):
        w = w_best.copy()
        w[i] = a                  # weight i and weight j change, the others stay at their best value
        w[j] = b
        return np.mean((Xs @ w + b_best - ys) ** 2)

    paths = [np.array(m.path) for m in models]

    # the range of the plot: covers all the paths and the best solution
    all_a = np.concatenate([p[:, i] for p in paths] + [[w_best[i]]])
    all_b = np.concatenate([p[:, j] for p in paths] + [[w_best[j]]])
    pad_a = 0.15 * (all_a.max() - all_a.min())
    pad_b = 0.15 * (all_b.max() - all_b.min())
    A, B = np.meshgrid(np.linspace(all_a.min() - pad_a, all_a.max() + pad_a, 50),
                       np.linspace(all_b.min() - pad_b, all_b.max() + pad_b, 50))

    Z = np.zeros_like(A)
    for r in range(A.shape[0]):
        for c in range(A.shape[1]):
            Z[r, c] = loss_2w(A[r, c], B[r, c])
    Z = np.log10(Z)

    colors = ["black", "magenta"]
    name_i, name_j = feature_cols[i], feature_cols[j]

    fig = plt.figure(figsize=(15, 6))

    # 3D surface
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    ax.plot_surface(A, B, Z, cmap="coolwarm", alpha=0.6)
    ax.contour(A, B, Z, zdir="z", offset=Z.min() - 0.3, cmap="coolwarm")       # contour lines on the floor
    for p, lab, col in zip(paths, labels, colors):
        z = np.log10([loss_2w(a, b) for a, b in zip(p[:, i], p[:, j])])
        ax.plot(p[:, i], p[:, j], z, color=col, marker=".", linewidth=2, label=lab)
    ax.scatter(0, 0, np.log10(loss_2w(0, 0)), color="green", s=80, label="start (all weights = 0)")
    ax.scatter(w_best[i], w_best[j], Z.min(), color="red", marker="*", s=200, label="best solution")
    ax.set_xlabel("weight of " + name_i)
    ax.set_ylabel("weight of " + name_j)
    ax.set_zlabel("log10(loss)")
    ax.set_title("Loss surface (3D)")
    ax.legend()

    # view from above
    ax2 = fig.add_subplot(1, 2, 2)
    cs = ax2.contourf(A, B, Z, levels=25, cmap="coolwarm")
    plt.colorbar(cs, ax=ax2, label="log10(loss)")
    for p, lab, col in zip(paths, labels, colors):
        ax2.plot(p[:, i], p[:, j], color=col, marker=".", markersize=4, linewidth=1.5, label=lab)
    ax2.plot(0, 0, "go", markersize=10, label="start (all weights = 0)")
    ax2.plot(w_best[i], w_best[j], "r*", markersize=18, label="best solution")
    ax2.set_xlabel("weight of " + name_i)
    ax2.set_ylabel("weight of " + name_j)
    ax2.set_title("View from above (the optimizer goes down to the center)")
    ax2.legend()

    plt.tight_layout()
    plt.savefig("minibatch_path.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_path([model_1, model_2], [L1, L2])

In [ ]:
# save the results in a file, the comparison notebook (06) will read it
def run_summary(label, m, losses):
    pred = m.predict(X_test)
    return {
        "label": label,
        "losses": [float(v) for v in losses],
        "epochs": [float(v) for v in m.epochs],
        "times": [float(v) for v in m.times],
        "train_mse": float(np.mean((m.predict(X_train) - y_train) ** 2)),
        "test_rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
        "test_r2": float(r2_score(y_test, pred)),
        "time": float(m.time),
    }

summary = {
    "name": "minibatch",
    "best_loss": float(best_loss),
    "normal_equation": {"train_mse": float(best_loss),
                        "test_rmse": float(np.sqrt(mean_squared_error(y_test, pred_exact))),
                        "test_r2": float(r2_score(y_test, pred_exact)),
                        "time": float(exact_time)},
    "runs": [run_summary(L1, model_1, losses_1), run_summary(L2, model_2, losses_2)],
}

with open("results_minibatch.json", "w") as f:
    json.dump(summary, f)
print("saved results_minibatch.json")

## Observations
_Write your own observations here after running the notebook. Some questions to help you:_

- _How close did each method get to the Normal Equation (look at "gap to best" and "distance to exact weights")?_
- _Which one converged faster, in epochs and in seconds? Are the two things the same?_
- _Is the loss curve smooth or noisy? Why?_
- _What does the path on the loss surface tell you about each method?_
- _Is the test RMSE close to what we expect from the training error? What does the R2 value tell you about the model?_
- _The test set is the last 2 years. Do prices in new years behave like the old years?_